# Lezione 10 — Correlazione e regressione lineare

**Corso:** Statistica Applicata con Python · **Blocco 4:** Modelli ed esperimenti
**Tempo stimato:** 120 minuti · **Capitolo del manuale:** `manual/10_correlacion_regresion_lineal_IT.md`

## Cosa vedrai in questa lezione

- **Covarianza** e **correlazione** di Pearson e di Spearman: cosa misurano e cosa non vedono.
- Quattro insiemi con **la stessa r** e forme molto diverse.
- **Correlazione non è causalità**: una variabile di confondimento simulata.
- La **regressione lineare ai minimi quadrati (OLS)**: calcolo a mano e con `statsmodels`.
- Lettura completa dell'*output*: coefficienti, errori standard, t, p-value, IC, R², R² corretto, F.
- Regressione **multipla** ("a parità delle altre"), una variabile **categoriale** (dummy) e la **previsione** con i suoi due intervalli.
- **Diagnostica dei residui** con dati reali di abitazioni della California (con una censura visibile), e cosa fare quando fallisce.

## Prerequisiti

- Lezioni 2 e 3 (media, deviazione standard, diagrammi di dispersione), 6 (errore standard, IC) e 7 (p-value, t-test).
- Librerie: `numpy`, `pandas`, `matplotlib`, `scipy`, `statsmodels` (`pip install numpy pandas matplotlib scipy statsmodels`; sono già in Google Colab).

> **Dati:** il notebook carica `datos/california_housing.csv` (5 000 distretti, campione casuale con seme 42 del dataset *California housing*, Pace e Barry, 1997, nella versione di scikit-learn). Se non lo trova, lo scarica con scikit-learn; e se nemmeno questo è possibile, genera un dataset **sintetico** con le stesse colonne (i numeri cambieranno). Gli esempi piccoli e le simulazioni sono *esempi illustrativi* sintetici.

---
## Cella 0 — Setup

Importiamo le librerie e fissiamo il **seme** del generatore casuale.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf

rng = np.random.default_rng(42)          # seme fisso: tutti otteniamo gli stessi risultati
plt.rcParams["figure.figsize"] = (8, 4)
plt.rcParams["axes.grid"] = True
print("Librerie caricate correttamente.")

---
## 1. Covarianza e correlazione a mano

Sei persone (*esempio illustrativo*): `horas` di pratica settimanale e `nota` in una prova. Calcoliamo la covarianza e la correlazione di Pearson passo per passo e verifichiamo con `numpy`.

In [ ]:
horas = np.array([1, 2, 3, 4, 5, 6], dtype=float)
nota = np.array([2, 3, 5, 4, 7, 7], dtype=float)

dx = horas - horas.mean()
dy = nota - nota.mean()
cov_xy = (dx * dy).sum() / (len(horas) - 1)             # covarianza campionaria (ddof=1)
r_mano = (dx * dy).sum() / np.sqrt((dx**2).sum() * (dy**2).sum())

print(f"Somma dei prodotti Sxy = {(dx * dy).sum():.3f}")
print(f"Covarianza = {cov_xy:.3f}")
print(f"r a mano = {r_mano:.4f}")
print(f"r np.corrcoef = {np.corrcoef(horas, nota)[0, 1]:.4f}")
print(f"r scipy pearsonr = {stats.pearsonr(horas, nota)[0]:.4f}")

**Osserva:** la covarianza (3,6) dipende dalle unità (ore × punti); r = 0,93 no: è sempre tra −1 e 1. Il fatto che sia positiva dice che a più ore corrisponde un voto maggiore **in questi sei dati**; non che le ore *causino* il voto.

---
## 2. Esperimento guidato: la stessa r, quattro storie diverse

Costruiamo quattro insiemi di 20 punti e **regoliamo ciascuno in modo che r = 0,80** esattamente: una nuvola lineare con rumore, una parabola, una retta perfetta con un valore anomalo (outlier) e due gruppi senza relazione interna. La funzione `ajustar` cerca con `brentq` il parametro che dà r = 0,80.

**Prima di eseguire, scrivi:** pensi che r = 0,80 descriva ugualmente bene tutti e quattro?

In [ ]:
from scipy.optimize import brentq

def r_de(x, y):
    return np.corrcoef(x, y)[0, 1]

xs = np.linspace(0, 10, 20)
ruido = rng.normal(0, 1, 20)
grupo = np.repeat([0, 1], 10)
x_grupos = np.concatenate([rng.normal(3, 0.8, 10), rng.normal(8, 0.8, 10)])
ruido_grupos = rng.normal(0, 1, 20)
x_atip = np.append(np.linspace(0, 9, 19), 10.0)

def forma_lineal(s):   return xs, 1 + 0.8 * xs + s * ruido
def forma_parabola(c): return xs, -(xs - c) ** 2
def forma_atipico(v):  return x_atip, np.append(1 + 0.8 * x_atip[:19] + 0.3 * ruido[:19], v)
def forma_grupos(d):   return x_grupos, ruido_grupos + d * grupo

def ajustar(funcion, lo, hi, r_obj=0.80):
    """Cerca il parametro che fa valere la correlazione r_obj."""
    p = brentq(lambda t: r_de(*funcion(t)) - r_obj, lo, hi)
    return funcion(p)

conjuntos = {
    "Nuvola lineare con rumore": ajustar(forma_lineal, 0.01, 30),
    "Parabola": ajustar(forma_parabola, 5.0001, 60),
    "Retta + un outlier": ajustar(forma_atipico, -500, 9),
    "Due gruppi": ajustar(forma_grupos, 0.01, 50),
}

fig, ejes = plt.subplots(1, 4, figsize=(14, 3.2))
for ax, (nombre, (x, y)) in zip(ejes, conjuntos.items()):
    b1, b0 = np.polyfit(x, y, 1)
    ax.scatter(x, y, color="steelblue", s=25)
    ax.plot(xs, b0 + b1 * xs, color="crimson")
    ax.set_title(f"{nombre}\nr = {r_de(x, y):.2f}", fontsize=9)
plt.tight_layout(); plt.show()

for nombre, (x, y) in conjuntos.items():
    print(f"{nombre:<32} Pearson = {r_de(x, y):.2f}   Spearman = {stats.spearmanr(x, y)[0]:.2f}")

### Cosa osservi?

- Le quattro r valgono 0,80, ma solo la prima è "una nuvola lineare". La **parabola** non è una retta: la relazione è perfetta e curva. Nella quarta, la correlazione viene **solo dalla separazione tra i gruppi**; dentro ciascun gruppo non c'è relazione.
- Un solo punto (l'outlier) abbassa r da quasi 1 a 0,80 (terzo grafico). Spearman, che usa i **ranghi**, ne è meno sensibile.
- Conclusione: **r riassume; non sostituisce il grafico.** Disegna sempre la nuvola prima di fidarti del numero.

---
## 3. Pearson contro Spearman: relazione monotona ma non lineare

Spearman è la correlazione di Pearson calcolata sui **ranghi**. Misura relazioni **monotone** (sempre crescenti o decrescenti), anche se non sono rette.

In [ ]:
x_m = np.arange(1, 11, dtype=float)
y_m = np.exp(x_m / 2)                     # crescita esponenziale: monotona, non lineare

rangos_x, rangos_y = stats.rankdata(x_m), stats.rankdata(y_m)
print(f"Pearson  = {r_de(x_m, y_m):.3f}")
print(f"Spearman = {stats.spearmanr(x_m, y_m)[0]:.3f}")
print(f"Pearson sui ranghi = {r_de(rangos_x, rangos_y):.3f}")

**Osserva:** Spearman vale esattamente 1 (l'ordine si conserva) e Pearson resta sotto perché la relazione non è una retta. Se Spearman è alto e Pearson basso, sospetta una relazione curva o un outlier.

---
## 4. Correlazione non è causalità: una variabile di confondimento

Simuliamo (*esempio illustrativo*) una **temperatura** `z` che aumenta sia le `ventas` di gelati `x` sia gli `incidentes` in spiaggia `y`. **Non c'è nessun effetto di `x` su `y`** nella simulazione: lo sappiamo perché l'abbiamo costruita così.

**Prima di eseguire:** pensi che `x` e `y` saranno correlate? Cosa succederà "controllando" per `z`?

In [ ]:
g = np.random.default_rng(7)                     # generatore proprio per questa simulazione
n_sim = 2000
z = g.normal(0, 1, n_sim)                      # temperatura (standardizzata)
x = z + g.normal(0, 1, n_sim)                  # vendite di gelati: dipende da z
y = z + g.normal(0, 1, n_sim)                  # incidenti in spiaggia: dipende da z, NON da x
sim = pd.DataFrame({"x": x, "y": y, "z": z})

print(f"Correlazione grezza di x con y: {r_de(x, y):.3f}")

m_crudo = smf.ols("y ~ x", data=sim).fit()
m_ajustado = smf.ols("y ~ x + z", data=sim).fit()
for nombre, m in [("y ~ x", m_crudo), ("y ~ x + z", m_ajustado)]:
    print(f"{nombre:<10} coef. di x = {m.params['x']:+.3f}   errore standard = {m.bse['x']:.3f}   p = {m.pvalues['x']:.2g}")

# Stratificare: correlazione di x con y dentro 10 fasce di temperatura
sim["tramo"] = pd.qcut(sim["z"], 10, labels=False)
r_tramos = [r_de(t["x"], t["y"]) for _, t in sim.groupby("tramo")]
print(f"Correlazione media di x con y dentro ciascuna fascia di temperatura: {np.mean(r_tramos):.3f}")

fig, ejes = plt.subplots(1, 2, figsize=(11, 4))
ejes[0].scatter(sim["x"], sim["y"], s=6, alpha=0.4)
ejes[0].set_title("Grezza: x e y sembrano legate"); ejes[0].set_xlabel("x"); ejes[0].set_ylabel("y")
sc = ejes[1].scatter(sim["x"], sim["y"], c=sim["z"], s=6, cmap="coolwarm")
ejes[1].set_title("Colorata per temperatura z"); ejes[1].set_xlabel("x")
plt.colorbar(sc, ax=ejes[1]); plt.tight_layout(); plt.show()

### Cosa osservi?

- La correlazione grezza è chiaramente positiva (≈ 0,5) e il coefficiente di `x` è "molto significativo"... **senza alcun effetto causale**.
- Aggiungendo `z` al modello, il coefficiente di `x` scende quasi a 0 e il suo p-value non è più piccolo. Dentro ciascuna fascia stretta di temperatura, la correlazione media di `x` con `y` è vicina a 0 (molto minore di quella grezza).
- Un'analisi osservazionale può "controllare" solo le variabili che **misura**. Se `z` non fosse nei dati, non vedremmo il problema. La causalità si stabilisce con il disegno (lezione 13), non con un coefficiente.

---
## 5. Minimi quadrati a mano

La retta `nota ≈ b0 + b1 · horas` che minimizza la **somma dei quadrati dei residui** ha pendenza `b1 = Sxy / Sxx` e intercetta `b0 = media(y) − b1 · media(x)`. Con i sei dati di prima:

In [ ]:
Sxx = (dx**2).sum()
Sxy = (dx * dy).sum()
b1 = Sxy / Sxx
b0 = nota.mean() - b1 * horas.mean()
pred = b0 + b1 * horas
resid = nota - pred

print(f"Sxx = {Sxx:.2f}   Sxy = {Sxy:.2f}")
print(f"b1 (pendenza) = {b1:.4f}   b0 (intercetta) = {b0:.4f}")
print("Residui:", np.round(resid, 3))
print(f"Somma dei residui (in valore assoluto) = {abs(resid.sum()):.3f}")
print(f"Somma dei quadrati dei residui = {(resid**2).sum():.4f}")
print("np.polyfit:", np.round(np.polyfit(horas, nota, 1), 4))

**Esperimento:** la pendenza `b1` è davvero quella che minimizza la somma dei quadrati? Proviamo altre pendenze (riposizionando l'intercetta in modo che la retta passi per il centro dei dati) e confrontiamo.

In [ ]:
pendientes = np.round(np.arange(0.6, 1.5, 0.1), 2)
filas = []
for p in pendientes:
    interc = nota.mean() - p * horas.mean()
    filas.append((p, ((nota - (interc + p * horas)) ** 2).sum()))
tabla_pend = pd.DataFrame(filas, columns=["pendenza", "SSR"])
print(tabla_pend.round(3).to_string(index=False))

grid = np.linspace(0.4, 1.7, 100)
ssr = [((nota - ((nota.mean() - p * horas.mean()) + p * horas)) ** 2).sum() for p in grid]
plt.plot(grid, ssr, color="steelblue")
plt.axvline(b1, color="crimson", linestyle="--", label=f"b1 = {b1:.3f}")
plt.xlabel("Pendenza provata"); plt.ylabel("Somma dei quadrati dei residui")
plt.title("La pendenza OLS è il minimo della parabola"); plt.legend(); plt.show()

**Cosa osservi?** La somma dei quadrati forma una parabola il cui minimo cade esattamente in `b1 = Sxy/Sxx`. "Minimi quadrati" significa questo: tra tutte le rette, quella scelta lascia la minor somma di errori al quadrato. Con due variabili, inoltre, R² = r² (0,93² ≈ 0,87).

---
## 6. Dati reali: abitazioni della California

Ogni riga è un **distretto** del censimento del 1990. `MedHouseVal` è il valore mediano delle abitazioni in **unità da 100 000 \$**; `MedInc`, il reddito mediano in decine di migliaia di \$; `HouseAge`, l'età mediana delle abitazioni; `AveRooms` e `AveOccup`, stanze e occupanti medi per abitazione. L'obiettivo è **censurato**: i valori alti sono troncati a 5,00001. Lo verifichiamo.

In [ ]:
def generar_california_sintetico(n=5000, seed=42):
    """Genera un dataset SINTETICO con le stesse colonne di California housing (alternativa se non c'è il CSV né il download).
    Le relazioni sono inventate: serve per esercitarsi, non per trarre conclusioni sulla California."""
    g = np.random.default_rng(seed)
    MedInc = np.clip(g.lognormal(1.25, 0.5, n), 0.5, 15)
    HouseAge = g.integers(1, 53, n).astype(float)
    AveRooms = np.clip(3.0 + 0.35 * MedInc + g.normal(0, 0.7, n), 2, 9)
    AveBedrms = np.clip(0.2 * AveRooms + g.normal(0, 0.05, n), 0.6, 2)
    Population = np.round(g.lognormal(7.0, 0.6, n))
    AveOccup = np.clip(g.lognormal(1.05, 0.25, n), 1.2, 5.5)
    Latitude = np.round(g.uniform(32.5, 42.0, n), 2)
    Longitude = np.round(g.uniform(-124.3, -114.5, n), 2)
    valor = (0.5 + 0.42 * MedInc + 0.012 * HouseAge - 0.15 * AveRooms - 0.3 * AveOccup
             + 0.05 * (37 - Latitude) + g.normal(0, 0.6, n))
    MedHouseVal = np.clip(valor, 0.15, 5.00001)          # censura a 5,00001, come l'originale
    return pd.DataFrame({"MedInc": MedInc, "HouseAge": HouseAge, "AveRooms": AveRooms, "AveBedrms": AveBedrms,
                         "Population": Population, "AveOccup": AveOccup, "Latitude": Latitude,
                         "Longitude": Longitude, "MedHouseVal": MedHouseVal})


def cargar_california():
    """Restituisce (DataFrame, origine): CSV locale, altrimenti download di scikit-learn, altrimenti dati sintetici."""
    for ruta in ("../datos/california_housing.csv", "datos/california_housing.csv", "california_housing.csv"):
        if os.path.exists(ruta):
            return pd.read_csv(ruta), "CSV locale"
    try:
        from sklearn.datasets import fetch_california_housing
        datos = fetch_california_housing(as_frame=True).frame
        return datos.sample(n=5000, random_state=42).reset_index(drop=True), "scikit-learn"
    except Exception:
        return generar_california_sintetico(), "sintetico"


df_bruto, origen = cargar_california()
print("Origine dei dati:", origen, "|", df_bruto.shape[0], "righe")

In [ ]:
# Preparazione: togliamo distretti con valori medi assurdi e creiamo la variabile di zona
df = df_bruto[(df_bruto["AveRooms"] < 10) & (df_bruto["AveOccup"] < 6)].copy()
df["zona"] = np.where(df["Latitude"] >= 36, "Nord", "Sud")
print(f"Distretti usati: {len(df)} su {len(df_bruto)}")

In [ ]:
print(df[["MedHouseVal", "MedInc", "HouseAge", "AveRooms", "AveOccup"]].describe().round(2).to_string())

techo = df["MedHouseVal"] >= 5
print(f"\nDistretti al tetto (MedHouseVal ≥ 5): {techo.sum()} ({techo.mean():.1%})")
print(f"Valore massimo: {df['MedHouseVal'].max()}")

fig, ejes = plt.subplots(1, 2, figsize=(11, 3.8))
ejes[0].hist(df["MedHouseVal"], bins=50, color="steelblue")
ejes[0].set_title("MedHouseVal: l'ultima barra è la censura"); ejes[0].set_xlabel("Valore (100 000 $)")
ejes[1].scatter(df["MedInc"], df["MedHouseVal"], s=4, alpha=0.3)
ejes[1].set_title("Valore rispetto al reddito mediano"); ejes[1].set_xlabel("MedInc"); ejes[1].set_ylabel("MedHouseVal")
plt.tight_layout(); plt.show()

print(f"\nCorrelazione di Pearson MedInc–MedHouseVal: {r_de(df['MedInc'], df['MedHouseVal']):.3f}")
print(f"Correlazione di Spearman MedInc–MedHouseVal: {stats.spearmanr(df['MedInc'], df['MedHouseVal'])[0]:.3f}")

**Osserva:** c'è una pila di distretti esattamente al tetto: sono valori **troncati**, non misurati. Nel diagramma di dispersione si vedono come una fascia orizzontale in alto. Torneremo su di essa nei residui.

---
## 7. Regressione semplice con `statsmodels` e lettura dell'output

`smf.ols("MedHouseVal ~ MedInc", data=df)` adatta la retta; `.fit()` calcola. Per avere un risultato riproducibile (senza data né ora) stampiamo la tabella dei coefficienti e le misure globali separatamente.

In [ ]:
def resumen(m):
    """Stampa la tabella dei coefficienti e le misure globali di un modello OLS."""
    print(m.summary().tables[1])
    print(f"n = {int(m.nobs)}   R² = {m.rsquared:.4f}   R² corretto = {m.rsquared_adj:.4f}   F = {m.fvalue:.1f}   p(F) = {m.f_pvalue:.2g}")

m1 = smf.ols("MedHouseVal ~ MedInc", data=df).fit()
resumen(m1)

In [ ]:
# Ricostruiamo a mano ciò che mostra la tabella
b0_m1, b1_m1 = m1.params["Intercept"], m1.params["MedInc"]
ee_m1 = m1.bse["MedInc"]
t_m1 = b1_m1 / ee_m1
gl = int(m1.df_resid)
t_crit = stats.t.ppf(0.975, gl)
print(f"t = coef / ee = {b1_m1:.4f} / {ee_m1:.4f} = {t_m1:.2f}")
print(f"IC 95 % a mano = ({b1_m1 - t_crit * ee_m1:.4f}, {b1_m1 + t_crit * ee_m1:.4f})   gradi di libertà = {gl}")
print(m1.conf_int().round(4).to_string())
print(f"r² di Pearson = {r_de(df['MedInc'], df['MedHouseVal'])**2:.4f}   R² del modello = {m1.rsquared:.4f}")
print(f"Effetto di +1 unità di MedInc (10 000 $): {b1_m1:+.3f} × 100 000 $ = {b1_m1 * 100_000:.0f} $")

### Cosa osservi?

- La pendenza è il cambiamento medio di `MedHouseVal` per ogni unità di `MedInc`; l'intercetta è il valore **previsto** con `MedInc = 0`, che è fuori dalla zona in cui ci sono dati: non interpretarla alla lettera.
- t = coef/ee e l'IC = coef ± t·ee coincidono con la tabella. Con migliaia di dati il p-value risulta "0,000": significa "minore di 0,0005", non "zero".
- Con una sola variabile, **R² = r²**. R² è la frazione della variabilità di `y` spiegata dalla retta, **non** "la probabilità di azzeccare".

---
## 8. Regressione multipla, variabile categoriale e R² corretto

Aggiungiamo `HouseAge`, `AveRooms`, `AveOccup` e la zona (`C(zona)`, con Nord come categoria di riferimento se `Latitude ≥ 36`, e Sud altrimenti). Ogni coefficiente si legge **a parità delle altre variabili**.

In [ ]:
m2 = smf.ols("MedHouseVal ~ MedInc + HouseAge + AveRooms + AveOccup + C(zona)", data=df).fit()
resumen(m2)

print(f"\nCorrelazione semplice AveRooms–MedHouseVal: {r_de(df['AveRooms'], df['MedHouseVal']):+.3f}")
print(f"Correlazione MedInc–AveRooms: {r_de(df['MedInc'], df['AveRooms']):+.3f}")
print(f"Coefficiente di AveRooms nel modello multiplo: {m2.params['AveRooms']:+.3f}")

### Cosa osservi?

- `AveRooms` ha correlazione **positiva** con il valore, ma il suo coefficiente nel modello multiplo è **negativo**. Non è un errore: a parità di reddito, occupazione ed età, più stanze medie si associano a un valore minore. `AveRooms` e `MedInc` sono correlate, e il modello semplice mescolava i due effetti.
- Il coefficiente della zona confronta **Sud con Nord** (la categoria di riferimento), a parità del resto.
- Interpretare un coefficiente senza il "a parità delle altre" è un errore frequente.

### Significativo non è lo stesso che importante

Un p-value piccolo dice che un coefficiente è **distinguibile da zero**, non che il suo effetto sia **grande**. Confrontiamo, in un modello con `Population`, l'effetto di una variazione di **una deviazione standard** in `MedInc` e in `Population`.

In [ ]:
m_pob = smf.ols("MedHouseVal ~ MedInc + HouseAge + AveRooms + AveOccup + Population", data=df).fit()
for v in ("MedInc", "Population"):
    efecto_sd = m_pob.params[v] * df[v].std()
    print(f"{v:<11} coef = {m_pob.params[v]:.6f}   p = {m_pob.pvalues[v]:.2g}   effetto di +1 dev. standard: {efecto_sd:+.3f} (×100 000 $)")

**Osserva:** entrambi i coefficienti hanno un p-value minuscolo, ma una variazione tipica di `Population` sposta il valore di una frazione di quanto lo sposta una variazione tipica di `MedInc`. Con migliaia di dati quasi tutto risulta "significativo": chiediti sempre **di quanto** cambia il risultato, non solo se il p-value è minore di 0,05.

### Esperimento guidato: R² sale sempre, R² corretto no

Prendiamo **100 distretti** e aggiungiamo colonne di **rumore puro** (numeri casuali senza alcuna relazione con nulla). Cosa succede a R² e a R² corretto? **Prima di eseguire**, scrivi la tua previsione.

In [ ]:
pequena = df.sample(100, random_state=1).reset_index(drop=True)
X_base = pequena[["MedInc", "HouseAge", "AveOccup"]]
filas = []
for k in (0, 5, 10, 20, 40):
    r2, r2aj = [], []
    for _ in range(200):                               # 200 ripetizioni con rumore diverso
        X_k = pd.concat([X_base, pd.DataFrame(rng.normal(0, 1, (100, k)), columns=[f"r{i}" for i in range(k)])], axis=1)
        m = sm.OLS(pequena["MedHouseVal"], sm.add_constant(X_k)).fit()
        r2.append(m.rsquared); r2aj.append(m.rsquared_adj)
    filas.append((k, np.mean(r2), np.mean(r2aj)))
print(pd.DataFrame(filas, columns=["variabili di rumore", "R² medio", "R² corretto medio"]).round(3).to_string(index=False))

### Cosa osservi?

R² **sale sempre** aggiungendo variabili, anche se sono puro rumore: con 40 colonne inutili e solo 100 righe sembra che il modello "spieghi" molto di più. L'R² corretto penalizza ogni variabile in più e non premia il rumore. Per questo **un R² alto non prova che il modello sia corretto**, e confrontare modelli con un numero diverso di variabili richiede l'R² corretto (o criteri come l'AIC, lezione 12).

---
## 9. Previsione: due intervalli diversi

Per un dato reddito `MedInc` ci sono due domande diverse:

- Qual è il **valore medio** dei distretti con quel reddito? → **IC della media** (stretto).
- Dove cadrà il valore di **un distretto specifico** con quel reddito? → **intervallo di previsione** (più ampio: aggiunge la dispersione individuale attorno alla retta).

Usiamo il modello semplice `m1`. Includiamo un valore **fuori dal range** dei dati per vedere perché estrapolare è pericoloso.

In [ ]:
nuevos = pd.DataFrame({"MedInc": [2.0, 4.0, 6.0, 10.0, 25.0]})
pf = m1.get_prediction(nuevos).summary_frame(alpha=0.05)
pf.insert(0, "MedInc", nuevos["MedInc"])
pf["ampiezza IC media"] = pf["mean_ci_upper"] - pf["mean_ci_lower"]
pf["ampiezza int. previsione"] = pf["obs_ci_upper"] - pf["obs_ci_lower"]
print(pf[["MedInc", "mean", "mean_ci_lower", "mean_ci_upper", "obs_ci_lower", "obs_ci_upper"]].round(2).to_string(index=False))
print(f"\nRange osservato di MedInc: {df['MedInc'].min():.2f} – {df['MedInc'].max():.2f}")

rejilla = pd.DataFrame({"MedInc": np.linspace(0.5, 12, 100)})
pg = m1.get_prediction(rejilla).summary_frame(alpha=0.05)
plt.scatter(df["MedInc"], df["MedHouseVal"], s=3, alpha=0.2, color="gray")
plt.plot(rejilla["MedInc"], pg["mean"], color="crimson", label="retta adattata")
plt.fill_between(rejilla["MedInc"], pg["mean_ci_lower"], pg["mean_ci_upper"], color="crimson", alpha=0.4, label="IC 95 % della media")
plt.plot(rejilla["MedInc"], pg["obs_ci_lower"], color="black", linestyle="--", linewidth=1, label="intervallo di previsione 95 %")
plt.plot(rejilla["MedInc"], pg["obs_ci_upper"], color="black", linestyle="--", linewidth=1)
plt.xlabel("MedInc"); plt.ylabel("MedHouseVal"); plt.legend(); plt.show()

### Cosa osservi?

- L'IC della media è molto stretto (con migliaia di dati conosciamo bene la retta); l'intervallo di previsione è **molto più ampio**: un singolo distretto varia molto attorno alla retta.
- Con `MedInc = 25`, fuori dal range osservato, la retta prevede un valore impossibile (ben oltre il tetto di 5). La relazione può non essere lineare oltre i dati: **non estrapolare**.
- Con `MedInc = 2` il limite inferiore dell'intervallo di previsione è negativo: un valore di abitazione negativo è impossibile, un altro segnale che il modello lineare è solo un'approssimazione.

---
## 10. Diagnostica dei residui

Un **residuo** è `y − ŷ`. Se il modello è adeguato, i residui sembrano rumore senza struttura. Guardiamo quattro grafici sul modello multiplo `m2`:

1. **Residui rispetto ai valori previsti:** deve vedersi una nuvola senza forma (linearità) e di ampiezza costante (omoschedasticità).
2. **Grafico QQ:** confronta i residui con una normale.
3. **Scala-localizzazione:** radice del residuo standardizzato in valore assoluto rispetto al previsto (un altro sguardo alla varianza).
4. **Leva rispetto alla distanza di Cook:** punti con molta influenza sull'adattamento.

Segniamo in rosso i distretti **censurati** (`MedHouseVal ≥ 5`).

In [ ]:
ajustados = m2.fittedvalues
residuos = m2.resid
infl = m2.get_influence()
estand = infl.resid_studentized_internal
palanca = infl.hat_matrix_diag
cook = infl.cooks_distance[0]
col = np.where(df["MedHouseVal"] >= 5, "crimson", "steelblue")

fig, ejes = plt.subplots(2, 2, figsize=(12, 8))
ejes[0, 0].scatter(ajustados, residuos, c=col, s=4, alpha=0.5)
ejes[0, 0].axhline(0, color="black", linewidth=1)
suav = sm.nonparametric.lowess(residuos, ajustados, frac=0.3)
ejes[0, 0].plot(suav[:, 0], suav[:, 1], color="orange", linewidth=2)
ejes[0, 0].set_title("1. Residui rispetto ai previsti"); ejes[0, 0].set_xlabel("Valore previsto"); ejes[0, 0].set_ylabel("Residuo")

sm.qqplot(estand, line="45", ax=ejes[0, 1], markersize=2)
ejes[0, 1].set_title("2. Grafico QQ (residui standardizzati)")

ejes[1, 0].scatter(ajustados, np.sqrt(np.abs(estand)), c=col, s=4, alpha=0.5)
ejes[1, 0].set_title("3. Scala-localizzazione"); ejes[1, 0].set_xlabel("Valore previsto"); ejes[1, 0].set_ylabel("√|residuo standardizzato|")

ejes[1, 1].scatter(palanca, cook, c=col, s=4, alpha=0.5)
ejes[1, 1].set_title("4. Leva rispetto alla distanza di Cook"); ejes[1, 1].set_xlabel("leverage"); ejes[1, 1].set_ylabel("Distanza di Cook")
plt.tight_layout(); plt.show()

techo = (df["MedHouseVal"] >= 5).values
print(f"Media dei residui in valore assoluto (sempre ≈ 0 in OLS con intercetta): {abs(residuos.mean()):.3f}")
print(f"Distretti censurati: {techo.sum()}   residuo medio in essi: {residuos[techo].mean():+.2f}   negli altri: {residuos[~techo].mean():+.2f}")
print(f"Asimmetria dei residui: {stats.skew(residuos):.2f}")
print(f"Maggiore distanza di Cook: {cook.max():.3f} ({'distretto'} {int(np.argmax(cook))})   soglia orientativa 4/n: {4 / len(df):.4f}")
print(f"Distretti con Cook > 4/n: {(cook > 4 / len(df)).sum()}")

### Cosa osservi?

- **Residui rispetto ai previsti:** i punti censurati (rosso) formano una **diagonale discendente** in alto: poiché il valore reale è troncato a 5,00001, il residuo è `5 − previsto`. È l'impronta della censura, un problema **reale** di questi dati. Inoltre la dispersione non è costante: la nuvola è più stretta a sinistra e più ampia al centro-destra (**eteroschedasticità**).
- **QQ:** la coda destra si stacca dalla diagonale: i residui sono asimmetrici, non normali.
- **Cook:** la maggior parte dei distretti ha distanza di Cook minuscola. Una soglia orientativa come 4/n ne segnala alcuni da **guardare**, non da cancellare automaticamente.
- **Importante:** la normalità dei residui **non è necessaria perché i coefficienti OLS siano buoni stimatori** (i minimi quadrati non la usano). Conta invece perché i **p-value e gli IC** siano affidabili quando il campione è **piccolo**; con migliaia di dati il TLC (lezione 5) attenua il problema, ma l'eteroschedasticità e la cattiva forma funzionale continuano a influire sull'inferenza.

---
## 11. Cosa fare quando la diagnostica segnala problemi

Tre risposte abituali, dalla più semplice alla più accurata:

1. **Errori standard robusti (HC3):** non cambiano i coefficienti, solo gli errori standard (e quindi t, p e IC), in modo che valgano anche se la varianza non è costante.
2. **Trasformare la risposta** (ad esempio `log(MedHouseVal)`): di solito stabilizza la varianza e riduce l'asimmetria; i coefficienti si leggono poi come variazioni **relative** approssimate.
3. **Cambiare la forma funzionale** o affrontare il problema di fondo (qui, la censura: un modello specifico per dati censurati, oltre questa lezione).

In [ ]:
from statsmodels.stats.diagnostic import het_breuschpagan

m2_hc3 = smf.ols("MedHouseVal ~ MedInc + HouseAge + AveRooms + AveOccup + C(zona)", data=df).fit(cov_type="HC3")
comparacion = pd.DataFrame({"coef": m2.params, "ee_OLS": m2.bse, "ee_HC3": m2_hc3.bse})
comparacion["ee_HC3 / ee_OLS"] = comparacion["ee_HC3"] / comparacion["ee_OLS"]
print(comparacion.round(4).to_string())

lm, lm_p, f_bp, f_p = het_breuschpagan(m2.resid, m2.model.exog)
print(f"\nTest di Breusch-Pagan (omoschedasticità): LM = {lm:.1f}, p = {lm_p:.2g}")

m_log = smf.ols("np.log(MedHouseVal) ~ MedInc + HouseAge + AveRooms + AveOccup + C(zona)", data=df).fit()
print(f"\nR² modello in scala originale = {m2.rsquared:.3f}   R² modello con log = {m_log.rsquared:.3f}   (non sono direttamente confrontabili)")
print(f"Asimmetria dei residui originale / log: {stats.skew(m2.resid):.2f} / {stats.skew(m_log.resid):.2f}")
print(f"Effetto approssimato di +1 in MedInc con il modello log: {100 * (np.exp(m_log.params['MedInc']) - 1):+.1f} %")

# Sensibilità: lo stesso modello senza i distretti censurati
m_sin = smf.ols("MedHouseVal ~ MedInc + HouseAge + AveRooms + AveOccup + C(zona)", data=df[df["MedHouseVal"] < 5]).fit()
print(f"\nCoefficiente di MedInc con tutti / senza censurati: {m2.params['MedInc']:.3f} / {m_sin.params['MedInc']:.3f}")

fig, ejes = plt.subplots(1, 2, figsize=(11, 4))
ejes[0].scatter(m2.fittedvalues, m2.resid, s=3, alpha=0.3); ejes[0].axhline(0, color="black", linewidth=1)
ejes[0].set_title("Modello originale"); ejes[0].set_xlabel("Previsto"); ejes[0].set_ylabel("Residuo")
ejes[1].scatter(m_log.fittedvalues, m_log.resid, s=3, alpha=0.3); ejes[1].axhline(0, color="black", linewidth=1)
ejes[1].set_title("Modello con log(MedHouseVal)"); ejes[1].set_xlabel("Previsto")
plt.tight_layout(); plt.show()

### Cosa osservi?

- Con errori robusti i coefficienti sono **identici**; cambiano gli errori standard (l'ultima colonna dice di quanto). Se il rapporto è vicino a 1, l'eteroschedasticità non influiva molto sull'inferenza.
- Breusch-Pagan rifiuta l'omoschedasticità, come suggeriva il grafico. Con 4 900 dati quasi qualsiasi piccola deviazione risulta "significativa": usalo come **avviso**, non come semaforo.
- Il logaritmo riduce l'asimmetria dei residui, ma **la censura resta** (il tetto esiste anche in scala log). Escludere i distretti censurati cambia la pendenza perché **taglia il campione in base al risultato**: nemmeno questa è una soluzione pulita. La diagnostica serve a **sapere quali limiti ha il tuo modello** e a dichiararli.

---
## Riepilogo della lezione

- **r** misura l'associazione **lineare** ed è sensibile agli outlier; **Spearman** misura l'associazione **monotona** con i ranghi. La stessa r può nascondere forme molto diverse: disegna sempre il grafico.
- **Correlazione ≠ causalità:** una variabile di confondimento crea correlazione senza effetto; si "controlla" solo ciò che si misura.
- **OLS** minimizza la somma dei quadrati dei residui; l'output dà coefficienti, errori standard, t, p-value, IC, R², R² corretto e F.
- Nella regressione multipla ogni coefficiente è **a parità delle altre**; R² sale sempre, R² corretto no.
- Ci sono due intervalli di previsione (media e previsione); **non estrapolare** fuori dal range dei dati.
- Diagnostica i residui: linearità, varianza, normalità (conta per l'**inferenza** con campioni piccoli, non per la stima), punti influenti. Con problemi: errori robusti (HC3), trasformazioni, altro modello.

**Prossima:** lezione 11 — regressione logistica e modelli lineari generalizzati (risposta binaria, odds ratio, VIF, AIC).